# Environment Validation Lab

Module: Setup and Python Ecosystem

## Lesson summary

This lab is the executable proof that the Module 0 environment is ready. It verifies the Python runtime, package versions, project paths, optional API credential variables, and reusable source helpers before the course starts using market data.

The lab should be run locally from JupyterLab launched with:

```bash
uv run jupyter lab
```

## Learning objectives

By the end of this lab, students should be able to:

- confirm that the active notebook kernel is using Python 3.12 or newer;
- verify that the scientific finance stack imports correctly;
- locate the project root and reusable `src/` helpers from any notebook working directory;
- check whether optional API credentials are available without printing secrets;
- produce a concise environment report for reproducibility.

A local environment is acceptable when the same inputs can reconstruct the same output:

$$
O = F(D, C, E, \theta).
$$

This lab mainly validates the environment term, $E$, and the project paths that make code, $C$, importable.



In [ ]:
import os
import platform
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path



In [ ]:
runtime_report = {
    "python": sys.version.split()[0],
    "implementation": platform.python_implementation(),
    "platform": platform.platform(),
}

assert sys.version_info >= (3, 12), runtime_report
runtime_report



## Package checks

In [ ]:
required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "statsmodels": "statsmodels",
    "sklearn": "scikit-learn",
    "arch": "arch",
    "plotly": "plotly",
    "ipywidgets": "ipywidgets",
    "yfinance": "yfinance",
    "jupyter_book": "jupyter-book",
}

package_report = {}
for import_name, distribution_name in required_packages.items():
    try:
        package_report[import_name] = version(distribution_name)
    except PackageNotFoundError:
        package_report[import_name] = "missing"

missing_packages = [
    name for name, package_version in package_report.items()
    if package_version == "missing"
]
assert not missing_packages, missing_packages
package_report



## Project paths

In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "_config.yml").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the current working directory.")


project_root = find_project_root(Path.cwd().resolve())
expected_paths = {
    "book_config": project_root / "_config.yml",
    "book_toc": project_root / "_toc.yml",
    "project_metadata": project_root / "pyproject.toml",
    "dependency_lock": project_root / "uv.lock",
    "source_helpers": project_root / "src",
    "class_notebooks": project_root / "notebooks" / "class",
    "generated_images": project_root / "img" / "generated",
}

path_report = {name: path.exists() for name, path in expected_paths.items()}
assert all(path_report.values()), path_report
path_report



## Local credential check

This cell checks whether optional variables exist, but never prints token values.

In [ ]:
credential_report = {
    "BANXICO_TOKEN": bool(os.environ.get("BANXICO_TOKEN")),
    "FRED_API_KEY": bool(os.environ.get("FRED_API_KEY")),
}

credential_report

## Source helper import

In [ ]:
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.market_data_quality import banxico_series_catalog, source_inventory_template
from src.time_series_diagnostics import adf_report

helper_report = {
    "banxico_catalog_rows": len(banxico_series_catalog()),
    "source_inventory_columns": list(source_inventory_template().columns),
    "adf_report_imported": callable(adf_report),
}
helper_report



## Environment report

The final report is intentionally compact. It records whether the runtime, packages, paths, credentials, and helper imports passed without exposing local secrets.



In [ ]:
environment_report = {
    "python_ok": sys.version_info >= (3, 12),
    "packages_ok": not missing_packages,
    "paths_ok": all(path_report.values()),
    "optional_credentials": credential_report,
    "helpers_ok": all(helper_report.values()),
}

environment_report

